# E2 · Condiciones de aplicabilidad (P2) → §9.2

**Pregunta (P2):** ¿los gates cierran la métrica cuando la regla no aplica y, sobre todo, no la dejan abierta
cuando no aplica?

Protocolo: §8.5.2 de la memoria y §6.2 de `experimentos.md`. No usa LLM ni API. Las `confianza` salen de los
**engines** del Nivel 2 ejecutados sobre C2, no de los `salida_*.json` de las ejecuciones. Así E2 no
arrastra el ruido de la transcripción del LLM, que es cosa de E3 y E4.

| Parte | Qué da | Necesita |
|---|---|---|
| **Medición** | `confianza`, motivo de cierre y variable de decisión de cada métrica en las 100 imágenes, con la réplica comprobada contra el engine | cachés de percepción y de gates |
| **Tabla 1** | % de C2 con cada métrica abierta, reparto de `fuente`, cobertura por dimensión y motivos de cierre | — |
| **Tabla 2** | Por gate: matriz 2×2 abierta/cerrada × aplica/no aplica, falsos positivos y falsos negativos con sus imágenes, y tasas con IC | A2 completa en las columnas de ese gate |

Hay **8 métricas con gate** pero **6 decisiones independientes**, porque `d_tercios`, `d_centro` y
`patron_dominante` comparten un único gate. `media_L` y `d_equilibrio` no tienen gate.

⚠️ **Anotación ciega.** La Tabla 2 de un gate solo se calcula cuando están completas sus columnas de A2: una
lista de fallos por imagen antes de terminar de anotar sesgaría la anotación. Por lo mismo, **no abras
`resultados/E2/gates_C2.csv` hasta terminar las columnas ciegas de A2**: dice qué métricas abre el sistema en
cada imagen.

### Cómo usarlo
1. Kernel: el Python de **`.venv`**. Ejecuta en orden.
2. La primera vez construye la caché de gates: los cuatro engines sobre C2, en un proceso aparte, en unos
   5 minutos. Se reconstruye sola si cambian el código, las constantes, las imágenes o la percepción.
3. Escribe en `experimentos/resultados/E2/`.

## 1. Configuración, criterio y congelación

In [ ]:
import sys, json
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ / "experimentos") not in sys.path:
    sys.path.insert(0, str(RAIZ / "experimentos"))
import comun
comun.preparar_sys_path()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown

CFG = comun.cargar_config()
CFG_E2 = CFG["E2"]
DEFINICION = CFG_E2["definicion_criterio"]
FP_MAX = CFG_E2["criterios"]["fp_max_por_gate"]
assert DEFINICION in ("estricta", "amplia")
RES = comun.dir_resultados("E2")

AZUL, NARANJA, TINTA, TINTA2, GRIS, REJILLA, EJE = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11,
    "axes.titlecolor": TINTA, "axes.labelcolor": TINTA2, "axes.edgecolor": EJE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": REJILLA, "grid.linewidth": 0.6, "xtick.color": GRIS, "ytick.color": GRIS,
    "lines.linewidth": 2, "legend.frameon": False,
})
pd.set_option("display.max_colwidth", 80)

PROC = comun.procedencia()
CONGELADO = comun.comprobar_congelacion()
print(f"commit {PROC['commit'][:7]} · congelado: {CONGELADO}")
print(f"Criterio: como mucho {FP_MAX} falsos positivos por gate, con la definición «{DEFINICION}» de «la regla aplica»")

## 2. Medición con los engines

`comun.construir_cache("gates")` ejecuta `_cache_gates.py` en un proceso aparte. Para cada imagen:

- llama a `analizar()` de los cuatro engines con la percepción de la caché (la misma que recibirían en una
  ejecución real);
- arbitra con `PrioridadEngine`, el código del crítico, leyendo esos informes. De ahí sale la **cobertura por
  dimensión** exacta que vería el crítico.

Las constantes de los gates se leen del código fuente (`comun.constantes_sistema`), así que la réplica usa
los mismos umbrales que el engine.

In [ ]:
comun.construir_cache("gates")
META_G = comun.cargar_cache("gates")["meta"]
G = comun.cargar_cache("gates")["imagenes"]
META_P, PERC, _ = comun.cargar_cache_percepcion()
STEMS = [p.stem for p in comun.imagenes_C2()]
assert list(G) == STEMS or sorted(G) == sorted(STEMS)

K = comun.constantes_sistema()
K_LUZ, K_LIN, K_ESP = K["tools/luz_tono_tool.py"], K["tools/lineas_direccion_tool.py"], K["tools/espacio_aislamiento_tool.py"]
IDENTIDAD = K["tools/composicion_espacial_tool.py"]["FUENTE_CON_IDENTIDAD"]
assert IDENTIDAD == K_ESP["FUENTE_CON_IDENTIDAD"] == "yolo"
print(f"Caché de gates del {META_G['fecha']} ({META_G['segundos']:.0f} s), sobre la percepción del {META_P['fecha']}")
print("Umbrales:", {"RATIO_CROMATICO_MIN": K_LUZ["RATIO_CROMATICO_MIN"],
                    "LONGITUD_MIN_HORIZONTE": K_LIN["LONGITUD_MIN_HORIZONTE"],
                    "ANGULO_MAX_HORIZONTE": K_LIN["ANGULO_MAX_HORIZONTE"], "MIN_LINEAS_FUGA": K_LIN["MIN_LINEAS_FUGA"],
                    "APERTURA_MIN_GRADOS": K_LIN["APERTURA_MIN_GRADOS"], "VAR_LAPLACIANO_MIN": K_ESP["VAR_LAPLACIANO_MIN"]})

### 2.1 Métricas y gates

Tabla larga, una fila por imagen y métrica, con la `confianza` del engine. Primero se comprueba que cada
informe tiene exactamente las métricas esperadas: si el contrato cambiara (una métrica nueva o renombrada),
el notebook falla en vez de dejarla fuera en silencio.

In [ ]:
GATES = {   # gate -> (dimensión, métricas que decide)
    "esquema_cromatico": ("luz_tono", ["esquema_cromatico"]),
    "angulo_horizonte": ("lineas_direccion", ["angulo_horizonte"]),
    "score_convergencia": ("lineas_direccion", ["score_convergencia"]),
    "espacio": ("espacio_aislamiento_sujeto", ["ratio_espacio_negativo"]),
    "nitidez": ("espacio_aislamiento_sujeto", ["ratio_nitidez"]),
    "composicion": ("composicion_espacial", ["d_tercios", "d_centro", "patron_dominante"]),
}
SIN_GATE = {"luz_tono": ["media_L"], "composicion_espacial": ["d_equilibrio"]}
GATE_DE = {m: g for g, (_, ms) in GATES.items() for m in ms}


def es_metrica(campo):     # misma definición que prioridad_tool.es_metrica
    return isinstance(campo, dict) and "valor" in campo and "confianza" in campo


esperadas = {d: sorted([m for g, (dd, ms) in GATES.items() if dd == d for m in ms] + SIN_GATE.get(d, []))
             for d in comun.DIMENSIONES}
filas = []
for s in STEMS:
    for d in comun.DIMENSIONES:
        inf = G[s]["informes"][d]
        metricas = {k: v for k, v in inf.items() if es_metrica(v)}
        assert sorted(metricas) == esperadas[d], f"{s} {d}: métricas {sorted(metricas)} != {esperadas[d]}"
        for m, v in metricas.items():
            filas.append({"imagen": s, "dimension": d, "metrica": m, "gate": GATE_DE.get(m, "(sin gate)"),
                          "confianza": v["confianza"], "abierta": v["confianza"] > 0,
                          "fuente_confianza": v["fuente_confianza"]})
M = pd.DataFrame(filas)
assert M[M.gate == "(sin gate)"].abierta.all(), "media_L y d_equilibrio deben estar siempre abiertas"
assert set(M.confianza) <= {0.0, 1.0}, f"Confianzas no binarias: {sorted(set(M.confianza))}"
print(f"{len(M)} filas = {len(STEMS)} imágenes × {M.metrica.nunique()} métricas")

### 2.2 Réplica de cada gate, motivo de cierre y variable de decisión

Para cada gate se recalcula la decisión a partir de los campos del informe y de la percepción, con las
mismas condiciones y en el mismo orden que el engine (`gate_cromatico`, `gate_horizonte`,
`gate_convergencia` y los gates de identidad y textura). El `assert` exige que coincida con la `confianza`
del engine en las 100 imágenes. El motivo es la **primera** condición que falla, como en el código.

Comprobaciones adicionales:

- los tres campos de composición comparten `confianza`;
- la máscara que recalcula el engine de espacio cubre lo mismo que la de la caché de percepción (GrabCut es
  determinista con `SEMILLA_GRABCUT`);
- la cobertura por dimensión del arbitraje coincide con `abiertas / métricas`.

In [ ]:
def decisiones(s):
    inf, p = G[s]["informes"], PERC[s]
    lt, ld, ea = inf["luz_tono"], inf["lineas_direccion"], inf["espacio_aislamiento_sujeto"]
    fuente = p["fuente"]
    identidad = fuente == IDENTIDAD
    filas = []

    ratio = lt["ratio_pixeles_cromaticos"]
    filas.append(("esquema_cromatico", ratio >= K_LUZ["RATIO_CROMATICO_MIN"],
                  "abierta" if ratio >= K_LUZ["RATIO_CROMATICO_MIN"] else "sin color analizable",
                  {"ratio_pixeles_cromaticos": ratio}))

    n_h, lon, ang = ld["n_lineas_horizonte"], ld["longitud_horizonte"], ld["angulo_horizonte"]["valor"]
    motivo = ("sin candidatos" if n_h == 0 else "fragmento corto" if lon < K_LIN["LONGITUD_MIN_HORIZONTE"]
              else "demasiado inclinado" if abs(ang) > K_LIN["ANGULO_MAX_HORIZONTE"] else "abierta")
    filas.append(("angulo_horizonte", motivo == "abierta", motivo,
                  {"n_lineas_horizonte": n_h, "longitud_horizonte": lon, "angulo": ang}))

    n_f, score, nulo, apertura = (ld["n_lineas_fuga"], ld["score_convergencia"]["valor"],
                                  ld["score_convergencia_nulo"], ld["apertura_haz"])
    motivo = ("pocas líneas" if n_f < K_LIN["MIN_LINEAS_FUGA"] else "no supera el azar" if score <= nulo
              else "haz casi paralelo" if apertura < K_LIN["APERTURA_MIN_GRADOS"] else "abierta")
    filas.append(("score_convergencia", motivo == "abierta", motivo,
                  {"n_lineas_fuga": n_f, "score": score, "score_nulo": nulo, "apertura": apertura}))

    filas.append(("espacio", identidad, "abierta" if identidad else f"sin identidad ({fuente})",
                  {"fuente": fuente}))

    var_max = max(ea["var_laplaciano_figura"], ea["var_laplaciano_fondo"])
    motivo = (f"sin identidad ({fuente})" if not identidad else
              "sin textura" if var_max < K_ESP["VAR_LAPLACIANO_MIN"] else "abierta")
    filas.append(("nitidez", motivo == "abierta", motivo, {"fuente": fuente, "var_max": var_max}))

    filas.append(("composicion", identidad, "abierta" if identidad else f"sin identidad ({fuente})",
                  {"fuente": fuente}))
    return [{"imagen": s, "gate": g, "abierta_replica": a, "motivo": m, **{f"v_{k}": v for k, v in var.items()}}
            for g, a, m, var in filas]


D = pd.DataFrame([f for s in STEMS for f in decisiones(s)])
abierta_engine = M[M.gate != "(sin gate)"].groupby(["imagen", "gate"]).abierta.agg(["min", "max"])
assert (abierta_engine["min"] == abierta_engine["max"]).all(), "las métricas de un mismo gate discrepan"
D = D.merge(abierta_engine["min"].rename("abierta").reset_index(), on=["imagen", "gate"])
discrepan = D[D.abierta != D.abierta_replica]
assert discrepan.empty, f"La réplica no coincide con el engine:\n{discrepan}"

# Máscara del engine de espacio frente a la de la caché de percepción.
dif_mascara = max(abs((1 - G[s]["informes"]["espacio_aislamiento_sujeto"]["ratio_espacio_negativo"]["valor"])
                      - PERC[s]["cobertura_mascara"]) for s in STEMS)
assert dif_mascara < 1e-4, f"La máscara recalculada difiere de la caché: {dif_mascara}"

# Cobertura del arbitraje frente a abiertas / métricas.
cob_replica = M.groupby(["imagen", "dimension"]).abierta.mean().round(4)
COB = pd.DataFrame({s: G[s]["arbitraje"]["cobertura_por_dimension"] for s in STEMS}).T[comun.DIMENSIONES]
assert all(abs(COB.at[s, d] - cob_replica[(s, d)]) < 1e-9 for s in STEMS for d in comun.DIMENSIONES)

print(f"Réplica OK: {len(D)} decisiones ({len(GATES)} gates × {len(STEMS)} imágenes) iguales al engine · "
      f"máscara de espacio igual a la caché (dif. máx. {dif_mascara:.1e}) · cobertura igual al arbitraje")

## 3. Tabla 1 · Qué abre el sistema en C2

Solo cifras agregadas, sin nada por imagen.

- **Tabla 1a:** % de C2 con cada métrica abierta, con IC de Wilson al 95 %.
- **Tabla 1b:** reparto de `fuente` en la percepción compartida. La rama `sin_sujeto_claro` no se da en C2, así
  que queda **sin validar con fotos reales** (§8.6).
- **Tabla 1c:** cobertura por dimensión (fracción de métricas abiertas), tal como la calcula el crítico.
- **Tabla 1d:** motivo de cierre por gate (la primera condición que falla).

In [ ]:
def proporcion(k, n):
    lo, hi = comun.intervalo_wilson(int(k), int(n))
    return pd.Series({"abiertas": int(k), "n": int(n), "%": 100 * k / n, "IC95 inf": 100 * lo, "IC95 sup": 100 * hi})


orden_metricas = [m for g in GATES for m in GATES[g][1]] + ["media_L", "d_equilibrio"]
T1A = (M.groupby("metrica").abierta.agg(["sum", "count"]).loc[orden_metricas]
       .apply(lambda r: proporcion(r["sum"], r["count"]), axis=1))
T1A = T1A.astype({"abiertas": int, "n": int})
T1A.insert(0, "gate", [GATE_DE.get(m, "(sin gate)") for m in T1A.index])
display(T1A.round(1))

fuentes = pd.Series({s: PERC[s]["fuente"] for s in STEMS}).value_counts()
T1B = fuentes.reindex(["yolo", "saliencia", "sin_sujeto_claro"], fill_value=0).rename("n").to_frame()
T1B["%"] = 100 * T1B.n / len(STEMS)
display(T1B)

T1C = pd.DataFrame({
    "cobertura media": COB.mean(),
    "n con cobertura 0": (COB == 0).sum(),
    "n con cobertura 1": (COB == 1).sum(),
    "valores posibles": [sorted(COB[d].unique().tolist()) for d in COB],
})
display(T1C.round(3))

T1D = (D.groupby(["gate", "motivo"]).size().rename("n").reset_index()
       .pivot(index="gate", columns="motivo", values="n").fillna(0).astype(int).loc[list(GATES)])
T1D = T1D[["abierta"] + [c for c in T1D.columns if c != "abierta"]]
T1D

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
y = np.arange(len(T1A))[::-1]
ax.hlines(y, T1A["IC95 inf"], T1A["IC95 sup"], color=AZUL, lw=2, alpha=0.45)
ax.plot(T1A["%"], y, "o", ms=8, color=AZUL, mec="white", mew=2)
for yi, (m, r) in zip(y, T1A.iterrows()):
    ax.text(r["IC95 sup"] + 1.5, yi, f"{r['%']:.0f} %", va="center", fontsize=8, color=TINTA2)
ax.set_yticks(y, [m if g == m else f"{m}  (sin gate)" if g == "(sin gate)" else f"{m}  (gate {g})"
                  for m, g in zip(T1A.index, T1A.gate)])
ax.set(xlim=(0, 112), xlabel="% de C2 con la métrica abierta (IC 95 % de Wilson)")
ax.set_title(f"Tabla 1 · Métricas abiertas en C2 (n = {len(STEMS)})", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(RES / "tabla1_abiertas.png", bbox_inches="tight")
plt.show()

## 4. Tabla 2 · Falsos positivos y falsos negativos

La verdad de referencia es A2 (`anotaciones/A2_aplicabilidad.csv`). Qué significa «la regla aplica» en cada
gate está en la guía de anotación y se codifica abajo, **en una sola función**, con sus dos versiones
(estricta y amplia). En la versión acotada (experimentos.md §15) las dos solo difieren en espacio y nitidez
(`segmentacion = parcial`); en el horizonte coinciden, porque no se anota `horizontal_estructural`.

Las anotaciones tienen cuatro estados: `si`, `no`, `dudoso` y vacío (pendiente). Se combinan con lógica de
tres valores: `no` y cualquier cosa da `no`; si no, un `dudoso` hace `dudoso`. Las imágenes cuyo resultado es
`dudoso` salen de la matriz 2×2 y se cuentan aparte.

Cada gate se calcula solo si están completas **todas** sus columnas de A2 y no hay revisiones de segmentación
obsoletas (`comun.revisiones_obsoletas`). Si no, se marca como pendiente.

In [ ]:
A2 = comun.leer_anotacion("A2_aplicabilidad", comun.ESQUEMA_A2)
assert sorted(A2.index) == sorted(STEMS)
OBSOLETAS = comun.revisiones_obsoletas(A2, PERC)
YOLO = [s for s in STEMS if PERC[s]["fuente"] == "yolo"]


def Y(*v):
    return "" if "" in v else "no" if "no" in v else "dudoso" if "dudoso" in v else "si"


def NO(v):
    return {"si": "no", "no": "si"}.get(v, v)


SEG = {"estricta": {"ok": "si", "parcial": "si", "mal": "no"}, "amplia": {"ok": "si", "parcial": "no", "mal": "no"}}


def aplica(s, gate, definicion):
    """¿La regla del gate aplica a la imagen s? Devuelve si / no / dudoso / '' (pendiente).
    Debe coincidir con la tabla de «Cómo se usan en E2» de GUIA_ANOTACION.md.
    Versión acotada: el horizonte no tiene definición amplia (no se anota `horizontal_estructural`)."""
    a, yolo = A2.loc[s], PERC[s]["fuente"] == "yolo"
    if gate == "esquema_cromatico":
        return a.color_analizable
    if gate == "angulo_horizonte":
        return a.horizonte_natural
    if gate == "score_convergencia":
        return a.perspectiva
    if gate in ("espacio", "nitidez"):
        seg = SEG[definicion].get(a.segmentacion, a.segmentacion) if yolo else "si"
        return Y(NO(a.sin_sujeto), seg)
    if gate == "composicion":
        return Y(NO(a.sin_sujeto), a.protagonista if yolo else "si")
    raise KeyError(gate)


COLUMNAS_GATE = {   # columnas de A2 que necesita cada gate (las de segmentación solo en las imágenes yolo)
    "esquema_cromatico": ["color_analizable"],
    "angulo_horizonte": ["horizonte_natural"],
    "score_convergencia": ["perspectiva"],
    "espacio": ["sin_sujeto", "segmentacion"],
    "nitidez": ["sin_sujeto", "segmentacion"],
    "composicion": ["sin_sujeto", "protagonista"],
}


def pendiente_de(gate):
    """Motivo por el que el gate aún no se puede evaluar, o None."""
    faltan = {}
    for col in COLUMNAS_GATE[gate]:
        filas = YOLO if col in comun.A2_SOLO_YOLO else STEMS
        n = int((A2.loc[filas, col] == "").sum())
        if n:
            faltan[col] = n
    if faltan:
        return "faltan " + ", ".join(f"{c} ({n})" for c, n in faltan.items())
    if gate in ("espacio", "nitidez", "composicion") and not OBSOLETAS.empty:
        return f"{len(OBSOLETAS)} revisiones de segmentación obsoletas"
    return None


PENDIENTE = {g: pendiente_de(g) for g in GATES}
pd.Series({g: p or "completa" for g, p in PENDIENTE.items()}, name="anotación").to_frame()

In [ ]:
def tabla_gate(gate, definicion):
    d = D[D.gate == gate].set_index("imagen")
    ap = pd.Series({s: aplica(s, gate, definicion) for s in STEMS})
    assert not (ap == "").any(), f"{gate}: quedan anotaciones pendientes"
    evaluables = ap[ap != "dudoso"].index
    abierta, si = d.loc[evaluables, "abierta"], ap[evaluables] == "si"
    fp = sorted(evaluables[abierta & ~si], key=lambda s: int(s[6:]))
    fn = sorted(evaluables[~abierta & si], key=lambda s: int(s[6:]))
    vp, vn = int((abierta & si).sum()), int((~abierta & ~si).sum())
    lo_fp, hi_fp = comun.intervalo_wilson(len(fp), len(fp) + vn)
    return {
        "gate": gate, "definicion": definicion, "n_evaluables": len(evaluables), "n_dudosas": int((ap == "dudoso").sum()),
        "VP": vp, "FP": len(fp), "FN": len(fn), "VN": vn,
        # Tasa de FP: de las imágenes donde la regla NO aplica, cuántas deja abiertas el gate.
        "tasa_FP": len(fp) / max(len(fp) + vn, 1), "tasa_FP_IC95": (lo_fp, hi_fp),
        # De las abiertas, cuántas no deberían estarlo (lo que vería un lector del informe).
        "FP_entre_abiertas": len(fp) / max(len(fp) + vp, 1),
        "tasa_FN": len(fn) / max(len(fn) + vp, 1),
        "imagenes_FP": fp, "imagenes_FN": fn, "imagenes_dudosas": sorted(ap.index[ap == "dudoso"], key=lambda s: int(s[6:])),
    }


RESULTADOS_T2 = [tabla_gate(g, dfn) for g in GATES if PENDIENTE[g] is None for dfn in ("estricta", "amplia")]
if RESULTADOS_T2:
    T2 = pd.DataFrame(RESULTADOS_T2).set_index(["gate", "definicion"])
    display(T2[["n_evaluables", "n_dudosas", "VP", "FP", "FN", "VN", "tasa_FP", "FP_entre_abiertas", "tasa_FN"]].round(3))
    display(T2[["imagenes_FP", "imagenes_FN"]])
else:
    T2 = pd.DataFrame()
    print("Tabla 2 pendiente en todos los gates:", {g: p for g, p in PENDIENTE.items()})

### 4.1 Hojas de los falsos positivos

Para cada falso positivo de la definición del criterio, la foto y el PNG de verificación de su dimensión (el
que genera el propio engine). Sirven para revisar el caso y para elegir los ejemplos de §9.5.

In [ ]:
def miniatura(ruta, lado=700):
    with Image.open(ruta) as im:
        im.draft("RGB", (lado, lado))
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((lado, lado))
        return np.asarray(im)


RUTA_IMG = {p.stem: p for p in comun.imagenes_C2()}
for r in RESULTADOS_T2:
    if r["definicion"] != DEFINICION or not r["imagenes_FP"]:
        continue
    dim = GATES[r["gate"]][0]
    casos = r["imagenes_FP"]
    fig, axs = plt.subplots(len(casos), 2, figsize=(9, 3.6 * len(casos)), squeeze=False)
    for (ax1, ax2), s in zip(axs, casos):
        verif = G[s]["informes"][dim]["verificacion_path"]
        ax1.imshow(miniatura(RUTA_IMG[s])); ax1.set_title(f"{s} · FP de {r['gate']}", loc="left", fontsize=9)
        if verif and Path(verif).exists():
            ax2.imshow(miniatura(verif)); ax2.set_title(f"verificación de {dim}", loc="left", fontsize=9)
        for ax in (ax1, ax2):
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(RES / f"fp_{r['gate']}.png", dpi=110, bbox_inches="tight")
    plt.show()

## 5. Criterio

Por gate, con la definición fijada en la configuración: **sí** si los falsos positivos no pasan de
`fp_max_por_gate`, **no** si pasan, y **pendiente** si falta anotación. El resultado global es **no** en cuanto
un gate falla, aunque otros sigan pendientes; es **sí** solo con los 6 gates evaluados y ninguno por encima.

In [ ]:
filas = []
for g in GATES:
    if PENDIENTE[g]:
        filas.append({"gate": g, "FP": None, "resultado": "pendiente", "nota": PENDIENTE[g]})
        continue
    r = T2.loc[(g, DEFINICION)]
    filas.append({"gate": g, "FP": int(r.FP), "resultado": "sí" if r.FP <= FP_MAX else "no",
                  "nota": ", ".join(r.imagenes_FP)})
CRITERIO = pd.DataFrame(filas).set_index("gate")
GLOBAL = ("no" if (CRITERIO.resultado == "no").any() else
          "pendiente" if (CRITERIO.resultado == "pendiente").any() else "sí")
display(Markdown(f"**Criterio global (≤ {FP_MAX} FP por gate, definición {DEFINICION}): {GLOBAL}**"))
CRITERIO

## 6. Exportación

En `experimentos/resultados/E2/`:

- `gates_C2.csv`: una fila por imagen y métrica, con la `confianza`, el motivo y las variables de decisión;
- `tabla1_abiertas.csv`, `tabla1_fuente.csv`, `tabla1_cobertura.csv` y `tabla1_motivos.csv`;
- `tabla2.csv` y `fp_fn.json` (solo los gates evaluables);
- `metricas.json`, con la procedencia y el criterio;
- las figuras.

In [ ]:
variables = D.drop(columns=["abierta_replica"]).rename(columns={"abierta": "abierta_gate"})
GATES_C2 = M.merge(variables, on=["imagen", "gate"], how="left")
GATES_C2.insert(1, "fuente_percepcion", GATES_C2.imagen.map(lambda s: PERC[s]["fuente"]))
GATES_C2.to_csv(RES / "gates_C2.csv", index=False, encoding="utf-8")
T1A.to_csv(RES / "tabla1_abiertas.csv", encoding="utf-8")
T1B.to_csv(RES / "tabla1_fuente.csv", encoding="utf-8")
T1C.to_csv(RES / "tabla1_cobertura.csv", encoding="utf-8")
T1D.to_csv(RES / "tabla1_motivos.csv", encoding="utf-8")
if not T2.empty:
    T2.drop(columns=["imagenes_FP", "imagenes_FN", "imagenes_dudosas"]).to_csv(RES / "tabla2.csv", encoding="utf-8")
comun.guardar_json({r["gate"] + "/" + r["definicion"]: {k: r[k] for k in ("imagenes_FP", "imagenes_FN", "imagenes_dudosas")}
                    for r in RESULTADOS_T2}, RES / "fp_fn.json")

metricas = {
    "experimento": "E2",
    "procedencia": PROC, "congelado": CONGELADO,
    "cache_gates": {k: META_G[k] for k in ("fecha", "commit", "segundos")},
    "cache_percepcion": {k: META_P[k] for k in ("fecha", "commit")},
    "umbrales": {"RATIO_CROMATICO_MIN": K_LUZ["RATIO_CROMATICO_MIN"], "LONGITUD_MIN_HORIZONTE": K_LIN["LONGITUD_MIN_HORIZONTE"],
                 "ANGULO_MAX_HORIZONTE": K_LIN["ANGULO_MAX_HORIZONTE"], "MIN_LINEAS_FUGA": K_LIN["MIN_LINEAS_FUGA"],
                 "APERTURA_MIN_GRADOS": K_LIN["APERTURA_MIN_GRADOS"], "VAR_LAPLACIANO_MIN": K_ESP["VAR_LAPLACIANO_MIN"],
                 "FUENTE_CON_IDENTIDAD": IDENTIDAD},
    "tabla1": {"abiertas": T1A.reset_index().to_dict("records"), "fuente": T1B.n.to_dict(),
               "cobertura_media": T1C["cobertura media"].to_dict(), "motivos": T1D.to_dict("index")},
    "tabla2": RESULTADOS_T2,
    "anotacion": {"pendiente_por_gate": PENDIENTE, "revisiones_obsoletas": OBSOLETAS.index.tolist()},
    "criterio": {"fp_max_por_gate": FP_MAX, "definicion": DEFINICION, "por_gate": CRITERIO.reset_index().to_dict("records"),
                 "global": GLOBAL},
}
comun.guardar_json(metricas, RES / "metricas.json")
sorted(p.name for p in RES.iterdir())

## 7. Limitaciones propias de E2 (para §8.6 y §9.2)

- **Anotador único:** A2 la anota una sola persona; no hay acuerdo entre anotadores.
- **Los umbrales se calibraron sobre este mismo C2** (capítulo 6): las tasas son optimistas respecto a fotos
  nuevas. Si se corrigen gates después de ver la Tabla 2, hay que declararlo y no presentarlo como validación.
- **`sin_sujeto_claro` no se ejercita:** ninguna imagen de C2 cae en esa rama (Tabla 1b).
- **YOLO elige la caja más confiable, no la protagonista:** el gate de identidad comprueba que hay un objeto
  reconocido, no que sea el sujeto. Los casos con `protagonista = no` lo muestran.
- **`ratio_nitidez` sin anotación de textura:** usa la misma definición de «aplica» que el espacio negativo, así
  que su condición de textura (`VAR_LAPLACIANO_MIN`) no se valida por separado.
- **n pequeño:** con 100 imágenes, un 0 % de falsos positivos es compatible con tasas reales de varios puntos
  (ver el IC de Wilson de la Tabla 2).